In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.handle_data import load_fd001
from sklearn.preprocessing import MinMaxScaler, FunctionTransformer
from tensorflow.keras.layers import Dense, LSTM, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, TensorBoard
from tensorflow.keras.models import Sequential
from tensorflow import keras
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
import os
import random
import numpy as np
import tensorflow as tf

# 1. Set Python environment variables
os.environ['PYTHONHASHSEED'] = '0'

# 2. Set Python built-in random seed
random.seed(42)

# 3. Set NumPy random seed
np.random.seed(42)

# 4. Set TensorFlow global random seed
tf.random.set_seed(42)


In [2]:
train_df, test_df, test_last = load_fd001()

In [3]:
train_df = train_df.drop(columns=["setting_3", "sensor_1", "sensor_10", "sensor_18", "sensor_19", 'sensor_5', 'sensor_16',])
test_df = test_df.drop(columns=["setting_3", "sensor_1", "sensor_10", "sensor_18", "sensor_19",'sensor_5', 'sensor_16'])
test_last = test_last.drop(columns=["setting_3", "sensor_1", "sensor_10", "sensor_18", "sensor_19", 'sensor_5', 'sensor_16',])

In [4]:
train_df["RUL"] = train_df["RUL"].clip(upper=125)

In [5]:
def add_rolling_features(df, sensor_cols, window=27):
    df = df.copy()
    df = df.sort_values(["engine_id", "cycle"]).copy()

    grouped = df.groupby("engine_id", sort=False)
    # df[f"{col}_roll_mean"] = grouped[col].transform(lambda s: s.rolling(window=window, min_periods=1).mean())
    # df[f"{col}_roll_std"] = grouped[col].transform(lambda s: s.rolling(window=window, min_periods=2).std())
    for col in sensor_cols:
        df[f"{col}_roll_mean"] = grouped[col].transform(
            lambda s: s.rolling(window=window, min_periods=1).mean()
        )

        df[f"{col}_roll_std"] = grouped[col].transform(
            lambda s: s.rolling(window=window, min_periods=2).std()
        )
    return df

In [6]:
def drop_id_col(df):
    df = df.copy()
    return df.drop(columns=["engine_id"])

In [7]:
sensor_cols = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']
rolling_transformer = FunctionTransformer(
    add_rolling_features,
    kw_args={
        "sensor_cols": sensor_cols,
        "window": 27
    },
    validate=False
)

In [8]:
pipeline = Pipeline([
    ("rolling_features", rolling_transformer),
    ("drop_id", FunctionTransformer(drop_id_col, validate=False)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

In [9]:
X_train = train_df.drop(columns=["RUL"])
y_train = train_df["RUL"]
X_test = test_df.sort_values(["engine_id", "cycle"]).groupby("engine_id").tail(1)
y_test = test_last["RUL"]

In [10]:
# pipeline.fit(X_train)
# X_train = pipeline.transform(X_train)
# X_test = pipeline.transform(X_test)

In [11]:
# feature_cols = [c for c in train_df.columns if c not in ["engine_id", "RUL"]]

In [12]:
# len(feature_cols)

In [13]:

def create_sequences(df, X, y=None, seq_length=30):
    """
    Create sequences independently for each engine.
    X must be a 2D array aligned row-by-row with df.
    """
    df = df.sort_values(["engine_id", "cycle"]).copy()

    # Preserve the same row order in X and y
    X = np.asarray(X)
    if y is not None:
        y = np.asarray(y)

    sequences = []
    targets = []

    for engine_id in df["engine_id"].unique():
        indices = np.flatnonzero(
            df["engine_id"].to_numpy() == engine_id
        )

        engine_X = X[indices]

        # Skip engines with fewer than seq_length cycles
        if len(engine_X) < seq_length:
            continue

        for end in range(seq_length, len(engine_X) + 1):
            sequences.append(engine_X[end - seq_length:end])

            if y is not None:
                targets.append(y[indices[end - 1]])

    sequences = np.asarray(sequences, dtype=np.float32)

    if y is not None:
        return sequences, np.asarray(targets, dtype=np.float32)

    return sequences



# Sort the original DataFrames
train_df = train_df.sort_values(
    ["engine_id", "cycle"]
).reset_index(drop=True)

test_df = test_df.sort_values(
    ["engine_id", "cycle"]
).reset_index(drop=True)

# Keep DataFrames separate from NumPy arrays
X_train_df = train_df.drop(columns=["RUL"]).copy()
y_train = train_df["RUL"].to_numpy(dtype=np.float32)

X_test_df = test_df.copy()

# Fit preprocessing ONLY on training data
X_train_scaled = pipeline.fit_transform(X_train_df)

# Transform test data with the same fitted pipeline
X_test_scaled = pipeline.transform(X_test_df)

# Build training sequences
X_train_seq, y_train_seq = create_sequences(
    X_train_df,
    X_train_scaled,
    y_train,
    seq_length=30
)

# Build test sequences
X_test_seq = []

for engine_id in X_test_df["engine_id"].unique():
    indices = np.flatnonzero(
        X_test_df["engine_id"].to_numpy() == engine_id
    )

    engine_X = X_test_scaled[indices]

    if len(engine_X) >= 30:
        X_test_seq.append(engine_X[-30:])

X_test_seq = np.asarray(X_test_seq, dtype=np.float32)

print("X_train_df type:", type(X_train_df))
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_train_seq shape:", X_train_seq.shape)
print("y_train_seq shape:", y_train_seq.shape)
print("X_test_seq shape:", X_test_seq.shape)

print("Training NaNs:", np.isnan(X_train_seq).sum())
print("Test NaNs:", np.isnan(X_test_seq).sum())


X_train_df type: <class 'pandas.DataFrame'>
X_train_scaled shape: (20631, 48)
X_train_seq shape: (17731, 30, 48)
y_train_seq shape: (17731,)
X_test_seq shape: (100, 30, 48)
Training NaNs: 0
Test NaNs: 0


In [14]:
model = Sequential()
model.add(Input(shape=(30, X_train_seq.shape[2])))
model.add(LSTM(64))
model.add(Dense(1, activation="linear"))
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="mse",
    metrics=[
        keras.metrics.MeanAbsoluteError(name="mae"),
        keras.metrics.R2Score(name="r2_score")
    ]
)

2026-10-23 22:28:12.554727: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5
2026-10-23 22:28:12.554757: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-10-23 22:28:12.554761: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
2026-10-23 22:28:12.554778: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-23 22:28:12.554791: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        28,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,993 (113.25 KB)

 Trainable params: 28,993 (113.25 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
history = model.fit(
    X_train_seq,
    y_train_seq,
    validation_split=0.15,
    epochs=100,
    batch_size=128,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5
        )
    ]
)

Epoch 1/100


2026-10-23 22:28:13.249469: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


118/118 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - loss: 6559.0264 - mae: 70.1530 - r2_score: -2.7698 - val_loss: 5940.4663 - val_mae: 66.2095 - val_r2_score: -2.3230 - learning_rate: 0.0010
Epoch 2/100
118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 5263.9595 - mae: 61.9831 - r2_score: -2.0255 - val_loss: 4992.9082 - val_mae: 60.2911 - val_r2_score: -1.7929 - learning_rate: 0.0010
Epoch 3/100
118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 4432.6904 - mae: 56.6410 - r2_score: -1.5477 - val_loss: 4241.3164 - val_mae: 55.5331 - val_r2_score: -1.3725 - learning_rate: 0.0010
Epoch 4/100
118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 3777.7729 - mae: 52.3676 - r2_score: -1.1713 - val_loss: 3645.2864 - val_mae: 51.7000 - val_r2_score: -1.0391 - learning_rate: 0.0010
Epoch 5/100
118/118 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 3259.6709 - mae: 48.9129 - r2_score: -0.8735 - val_loss: 3172.9041 - val_mae: 48.6040 - val_r2_score: -0.7749 - learning_rate: 0.0010
Epoch 6/100
118/118 ━━━━━━━━━━━━

In [17]:

# Check raw features and target
print("Raw feature NaNs:", X_train_df.isna().sum().sum())
print("Raw target NaNs:", np.isnan(y_train).sum())

# Check transformed features
print("Scaled NaNs:", np.isnan(X_train_scaled).sum())
print("Scaled infinities:", np.isinf(X_train_scaled).sum())

# Check sequences and labels
print("Sequence NaNs:", np.isnan(X_train_seq).sum())
print("Sequence infinities:", np.isinf(X_train_seq).sum())
print("Target NaNs:", np.isnan(y_train_seq).sum())
print("Target infinities:", np.isinf(y_train_seq).sum())

print("Feature range:",
      np.nanmin(X_train_seq),
      np.nanmax(X_train_seq))
print("Target range:",
      np.nanmin(y_train_seq),
      np.nanmax(y_train_seq))


Raw feature NaNs: 0
Raw target NaNs: 0
Scaled NaNs: 0
Scaled infinities: 0
Sequence NaNs: 0
Sequence infinities: 0
Target NaNs: 0
Target infinities: 0
Feature range: 0.0 1.0
Target range: 0.0 125.0


In [18]:
# print("X_train:", X_train.shape)
# print("y_train:", y_train.shape)
# print("Expected input:", (30, len(feature_cols)))

In [19]:
y_train_pred = model.predict(X_train_seq, verbose=0).ravel()

# Testing predictions
y_test_pred = model.predict(X_test_seq, verbose=0).ravel()

# Actual targets
y_train_true = y_train_seq.ravel()
y_test_true = test_last.sort_values("engine_id")["RUL"].to_numpy()

# If you trained on normalized targets, inverse-transform predictions
# and actual training targets before calculating metrics.

def calculate_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

results = pd.DataFrame({
    "Training": calculate_metrics(y_train_true, y_train_pred),
    "Testing": calculate_metrics(y_test_true, y_test_pred)
}).T

In [20]:
print(results.round(4))

             MAE       MSE     RMSE      R2
Training  4.8271   51.4320   7.1716  0.9706
Testing   9.7439  181.9285  13.4881  0.8946


In [24]:
def nasa_score(y_true, y_pred):
    errors = np.asarray(y_pred) - np.asarray(y_true)

    scores = np.where(
        errors < 0,
        np.exp(-errors / 13) - 1,
        np.exp(errors / 10) - 1
    )

    return scores.sum()

print(f"NASA score: {nasa_score(y_test_true, y_test_pred):.2f}")
# score = nasa_score(y_test_true, y_test_pred)
# print(f"NASA score: {score:.2f}")

NASA score: 299.75


In [26]:
model.save("lstm_model.keras")